# 11 Genre Prediction from Spotify Audio Features

This notebook builds genre classifier und predicter. This will be used to evaluate the recommender models built in other notebooks.

In [1]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

processed_dir = PROJECT_ROOT / 'data' / 'processed'

tracks = pd.read_parquet(processed_dir / 'tracks_clean.parquet')
artists = pd.read_parquet(processed_dir / 'artists_clean.parquet')
audio_features = pd.read_parquet(processed_dir / 'audio_features_clean.parquet')

print(len(tracks), len(artists), len(audio_features))

95977 56129 95948


## Merge datasets

In [2]:
"""
Merge tracks + artists, then train a classifier to fill in missing genres.

The core problem with naive merges on this kind of data:
- tracks.artists_id is a STRING that looks like a list, e.g. "['3mxJ...']"
- artists.genres is also a stringified list, e.g. "['pop', 'rock']"
- A track can have multiple artists, each with different genres
Pandas treats both as plain text unless you explicitly parse them, so a
direct merge on artists_id == artists.id will never match anything.

Pipeline:
1. Parse stringified lists into real Python lists (ast.literal_eval).
2. Explode tracks so there's one row per (track, artist) pair.
3. Merge that against artists on the artist id.
4. Re-aggregate genres back up to one row per track (union of all its
   artists' genres), since a track can have N artists.
5. Split into "has genre" (training data) vs "empty genre" (rows to predict).
6. Train a multi-label classifier (a track can have multiple genres) using
   audio/track features, then predict genres for the empty rows.
"""

import ast
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report

# ---------- Step 1: parse stringified lists ----------
def parse_list_string(s):
    """Turn "['pop', 'rock']" into ['pop', 'rock']. Handles NaN and junk safely."""
    if pd.isna(s):
        return []
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return [v for v in val if v]  # drop empty strings inside the list
        return []
    except (ValueError, SyntaxError):
        return []


tracks["artist_id_list"] = tracks["artists_id"].apply(parse_list_string)
artists["genre_list"] = artists["genres"].apply(parse_list_string)

# Sanity check before continuing -- if this prints mostly 0s/1s something
# upstream is still wrong (e.g. wrong column, or not actually list-strings).
print("Tracks with at least one parsed artist id:",
      (tracks["artist_id_list"].apply(len) > 0).mean())
print("Artists with at least one parsed genre:",
      (artists["genre_list"].apply(len) > 0).mean())


# ---------- Step 2: explode tracks to one row per (track, artist) ----------
exploded = tracks.explode("artist_id_list").rename(columns={"artist_id_list": "artist_id"})


# ---------- Step 3: merge against artists on artist id ----------
# Use artists' real id column, not its track_id column -- track_id is for a
# different relationship (artist -> their tracks) and isn't the join key here.
merged = exploded.merge(
    artists[["id", "genre_list"]],
    left_on="artist_id",
    right_on="id",
    how="left",
    suffixes=("", "_artist"),
)


# ---------- Step 4: re-aggregate to one row per track ----------
agg_genres = (
    merged.groupby("id")["genre_list"]
    .apply(lambda lists: sorted(set(g for sub in lists for g in (sub if isinstance(sub, list) else []))))
)

tracks_final = tracks.drop(columns=["artist_id_list"]).merge(
    agg_genres.rename("genre_list"), on="id", how="left"
)

print(f"\nTotal tracks: {len(tracks_final)}")
print(f"Tracks with at least one genre: {(tracks_final['genre_list'].apply(len) > 0).sum()}")
print(f"Tracks with empty genre: {(tracks_final['genre_list'].apply(len) == 0).sum()}")

Tracks with at least one parsed artist id: 1.0
Artists with at least one parsed genre: 0.5806089543729623

Total tracks: 95977
Tracks with at least one genre: 80448
Tracks with empty genre: 15529


## Mapping
To get the number of different genres down and to make training a prediction model easier, we map subgenres into bigger groups.

In [3]:
import re

def normalize(g):
    g = str(g).lower()
    g = g.replace("-", " ")
    g = re.sub(r"[^a-z0-9\s]", "", g)
    g = re.sub(r"\s+", " ", g).strip()
    return g

In [4]:
ROOTS = {
    "hip_hop": [
        "hip hop", "hip-hop", "hiphop",
        "rap", "trap", "drill", "grime",
        "boom bap", "cloud rap", "emo rap",
        "gangsta rap", "conscious rap",
        "trap soul", "melodic rap", "uk drill",
        "atl trap", "rap music", "hiphop music"
    ],

    "pop": [
        "pop", "pop music",
        "dance pop", "synth pop", "synthpop",
        "electropop", "indie pop", "alt pop",
        "teen pop", "hyperpop", "art pop",
        "k-pop", "j-pop", "europop",
        "bedroom pop", "power pop"
    ],

    "rock": [
        "rock", "rock music",
        "alternative rock", "alt rock",
        "indie rock", "classic rock",
        "garage rock", "hard rock",
        "post rock", "psychedelic rock",
        "punk rock", "post punk",
        "emo rock", "grunge", "britpop"
    ],

    "metal": [
        "metal", "metal music",
        "heavy metal", "death metal",
        "black metal", "doom metal",
        "thrash metal", "metalcore",
        "deathcore", "djent", "progressive metal",
        "nu metal", "power metal"
    ],

    "electronic": [
        "electronic", "edm", "electro",
        "house", "deep house", "tech house",
        "techno", "minimal techno", "acid techno",
        "trance", "progressive trance",
        "dubstep", "brostep", "drum and bass",
        "dnb", "garage", "uk garage",
        "ambient", "downtempo", "future bass",
        "glitch", "idm", "hardstyle",
        "electronic music", "club music"
    ],

    "rnb_soul": [
        "r&b", "rnb", "rhythm and blues",
        "soul", "neo soul", "funk",
        "contemporary r&b", "quiet storm",
        "motown", "soul music", "funk soul"
    ],

    "jazz": [
        "jazz", "jazz music",
        "bebop", "swing", "fusion",
        "smooth jazz", "free jazz",
        "latin jazz", "jazz fusion",
        "cool jazz"
    ],

    "classical": [
        "classical", "classical music",
        "orchestra", "symphony",
        "baroque", "opera", "chamber music",
        "concerto", "piano classical",
        "modern classical", "film score"
    ],

    "latin": [
        "latin", "latin music",
        "reggaeton", "bachata", "salsa",
        "cumbia", "latin pop", "urbano",
        "latin trap", "merengue", "banda",
        "regional mexican"
    ],

    "country": [
        "country", "country music",
        "americana", "bluegrass",
        "alt country", "country pop",
        "outlaw country", "honky tonk",
        "folk country"
    ],

    "folk_acoustic": [
        "folk", "folk music",
        "acoustic", "singer-songwriter",
        "indie folk", "americana folk",
        "roots", "traditional folk",
        "neo folk", "alt folk",
        "acoustic pop"
    ],

    "experimental": [
        "experimental", "experimental music",
        "avant-garde", "avant garde",
        "noise", "noise music",
        "industrial", "glitch",
        "idm", "electroacoustic",
        "contemporary experimental"
    ],

    "lofi": [
        "lofi", "lo-fi", "lo fi",
        "lo-fi beats", "chillhop",
        "study beats", "lofi hip hop",
        "ambient lofi"
    ]
}

In [5]:
def map_to_root(genres):
    if not isinstance(genres, list):
        return ["other"]

    genres = [normalize(g) for g in genres]

    result = set()

    for g in genres:
        matched = False

        for root, keywords in ROOTS.items():
            for k in keywords:
                if k in g:
                    result.add(root)
                    matched = True
                    break
            if matched:
                break

        if not matched:
            result.add("other")

    return list(result - {"unknown_candidate"})

In [6]:
def fallback_guess(g):
    if "rock" in g: return "rock"
    if "rap" in g or "hip" in g: return "hip_hop"
    if "pop" in g: return "pop"
    if "house" in g or "techno" in g: return "electronic"
    return "other"

In [27]:
tracks_final["genre_canonical"] = tracks_final["genre_list"].apply(map_to_root)

has_genre = tracks_final[
    tracks_final["genre_canonical"].apply(len) > 0
]
missing_genre = tracks_final[
    tracks_final["genre_canonical"].apply(lambda x: len(x) == 0 )
].copy()


In [28]:
feature_cols = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
]
feature_cols = [c for c in feature_cols if c in tracks_final.columns]

PARENT_GENRES = set(ROOTS.keys())

def is_valid_label(genres):
    return (
        isinstance(genres, list)
        and len(genres) > 0
        and any(g in PARENT_GENRES for g in genres)
    )

train_mask = tracks_final["genre_canonical"].apply(is_valid_label)

train_df = tracks_final[train_mask].copy()

y_raw = train_df["genre_canonical"].apply(
    lambda xs: [x for x in xs if x in PARENT_GENRES]
)

mlb = MultiLabelBinarizer(classes=sorted(PARENT_GENRES))
y = mlb.fit_transform(y_raw)
X = train_df[feature_cols].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [29]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

clf = OneVsRestClassifier(
    LogisticRegression(
        solver="saga",
        max_iter=1000
    )
)

clf.fit(X_train, y_train)

,estimator,LogisticRegre...solver='saga')
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None


In [30]:
y_pred = clf.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    target_names=mlb.classes_,
    zero_division=0
))

               precision    recall  f1-score   support

    classical       0.83      0.73      0.78       759
      country       0.00      0.00      0.00       602
   electronic       0.71      0.22      0.33      2003
 experimental       0.00      0.00      0.00       116
folk_acoustic       0.00      0.00      0.00       881
      hip_hop       0.73      0.37      0.49      2845
         jazz       0.00      0.00      0.00       750
        latin       0.00      0.00      0.00      1440
         lofi       0.00      0.00      0.00       222
        metal       0.66      0.15      0.25       515
          pop       0.64      0.82      0.72      7002
     rnb_soul       0.00      0.00      0.00      1019
         rock       0.64      0.16      0.25      3380

    micro avg       0.66      0.39      0.49     21534
    macro avg       0.32      0.19      0.22     21534
 weighted avg       0.51      0.39      0.40     21534
  samples avg       0.57      0.43      0.47     21534



In [31]:
infer_mask = ~train_mask
infer_df = tracks_final[infer_mask].copy()

X_missing = infer_df[feature_cols].fillna(0)

pred = clf.predict(X_missing)
infer_df["predicted_genre_list"] = mlb.inverse_transform(pred)

train_df["predicted_genre_list"] = train_df["genre_canonical"]
tracks_with_predicted_genres = pd.concat(
    [train_df, infer_df],
    ignore_index=True
)
tracks_with_predicted_genres.to_parquet(
    processed_dir / "tracks_with_predicted_genres.parquet",
    compression="snappy",
    index=False
)